# 10장. 분류 분석으로 주문 취소 여부 예측하기

이 노트북은 완료 주문과 취소 주문만 사용해 `is_cancelled`를 예측하는 이진 분류 실습입니다.

## 학습 목표

- `completed=0`, `cancelled=1`로 타깃 범위를 명확하게 정의합니다.
- `refunded`와 기타 상태를 이진 분류 대상에서 제외합니다.
- 병합의 키 관계, 행 수, 미매칭을 검증합니다.
- train, validation, test를 분리합니다.
- Dummy 기준 모델, Logistic Regression, Random Forest를 비교합니다.
- validation에서 모델과 임계값을 선택하고 test는 최종 평가에 한 번만 사용합니다.
- 예측 결과에서 고객명 등 불필요한 개인정보를 제외합니다.


## 1. 프로젝트 루트와 실행 환경 설정

전처리 파일이 없다면 프로젝트 루트에서 다음 명령을 실행합니다.

```bash
python scripts/prepare_ch10_data.py
```

이 실행에는 프로젝트의 `src.data_loader`와 `src.preprocessing` 모듈이 필요합니다. 이번 분석은 첨부된 `data/processed/` 파일을 사용했습니다.


In [1]:
from pathlib import Path
import sys

import pandas as pd


def find_project_root(start_path):
    start_path = Path(start_path).resolve()
    for candidate in [start_path, *start_path.parents]:
        if (candidate / 'requirements.txt').exists() and (candidate / 'scripts').exists():
            return candidate
    raise FileNotFoundError('프로젝트 루트 폴더를 찾을 수 없습니다.')


PROJECT_ROOT = find_project_root(Path.cwd())
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
REPORT_DIR = PROJECT_ROOT / 'reports'
REPORT_DIR.mkdir(parents=True, exist_ok=True)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print('Python 실행 파일:', sys.executable)
print('프로젝트 루트:', PROJECT_ROOT)
print('전처리 데이터 폴더:', PROCESSED_DIR)
print('보고서 폴더:', REPORT_DIR)


Python 실행 파일: c:\Users\ojh23\Documents\projects\llm-data-analysis-course\.venv\Scripts\python.exe
프로젝트 루트: C:\Users\ojh23\Documents\projects\llm-data-analysis-course
전처리 데이터 폴더: C:\Users\ojh23\Documents\projects\llm-data-analysis-course\data\processed
보고서 폴더: C:\Users\ojh23\Documents\projects\llm-data-analysis-course\reports


## 2. 전처리 데이터 확인과 불러오기

이 장에서는 `customers_clean.csv`, `orders_clean.csv`, `order_items_clean.csv`를 사용합니다.


In [2]:
required_files = [
    PROCESSED_DIR / 'customers_clean.csv',
    PROCESSED_DIR / 'orders_clean.csv',
    PROCESSED_DIR / 'order_items_clean.csv',
]

missing_files = [path for path in required_files if not path.exists()]
if missing_files:
    raise FileNotFoundError(
        '전처리 파일이 없습니다: '
        + ', '.join(str(path) for path in missing_files)
        + '. 먼저 python scripts/preprocess_data.py를 실행하세요.'
    )

customers = pd.read_csv(PROCESSED_DIR / 'customers_clean.csv')
orders = pd.read_csv(PROCESSED_DIR / 'orders_clean.csv')
order_items = pd.read_csv(PROCESSED_DIR / 'order_items_clean.csv')

print('customers:', customers.shape, customers.columns.tolist())
print('orders:', orders.shape, orders.columns.tolist())
print('order_items:', order_items.shape, order_items.columns.tolist())


customers: (150, 6) ['customer_id', 'name', 'gender', 'age', 'city', 'signup_date']
orders: (300, 7) ['order_id', 'customer_id', 'order_date', 'payment_method', 'order_status', 'order_month', 'order_dayofweek']
order_items: (764, 6) ['order_item_id', 'order_id', 'product_id', 'quantity', 'unit_price', 'line_total']


## 3. 주문 상태와 타깃 범위 확인

이번 실습은 완료 주문과 취소 주문만 비교합니다.

- `completed` → 0
- `cancelled` → 1
- `refunded`와 기타 상태 → 학습 대상에서 제외


In [3]:
status_summary = (
    orders['order_status']
    .value_counts(dropna=False)
    .rename_axis('order_status')
    .reset_index(name='order_count')
)
status_summary


,order_status,order_count
0,completed,184
1,cancelled,64
2,refunded,52


## 4. 분류 데이터와 병합 검증표 생성

공통 함수는 주문 특징 집계, 안전한 병합, 시간 순서 점검, 타깃 생성까지 수행합니다.


In [4]:
from src.classification import (
    build_classification_dataset,
    target_distribution,
)

(
    model_data,
    numeric_features,
    categorical_features,
    merge_checks,
    data_quality_checks,
) = build_classification_dataset(
    customers=customers,
    orders=orders,
    order_items=order_items,
)

target_dist = target_distribution(model_data)

print('모델링 데이터:', model_data.shape)
print('숫자형 feature:', numeric_features)
print('범주형 feature:', categorical_features)
display(target_dist)
display(merge_checks)
display(data_quality_checks)


모델링 데이터: (248, 16)
숫자형 feature: ['age', 'item_count', 'total_quantity', 'order_amount', 'order_month', 'order_dayofweek', 'days_since_signup']
범주형 feature: ['gender', 'city', 'payment_method']


,is_cancelled,class_label,count,ratio
0,0,completed,184,0.7419
1,1,cancelled,64,0.2581


,merge,before_rows,after_rows,row_count_preserved,unmatched_count,status
0,order_id → order_item_features,248,248,True,0,PASS
1,customer_id → customers,248,248,True,0,PASS


,check_item,count
0,binary_target_rows,248
1,completed_rows,184
2,cancelled_rows,64
3,excluded_status_rows,52
4,line_total_mismatch,0
5,signup_order_date_reversal_rows,45
6,days_since_signup_missing_after_policy,45
7,missing_order_item_features,0
8,missing_customer_match,0
9,status_scope:completed,184


## 5. 데이터 누수 확인

`order_status`와 `is_cancelled`는 입력 feature에 포함되면 안 됩니다.


In [5]:
features = numeric_features + categorical_features
leakage_columns = {'order_status', 'is_cancelled'}
leakage_found = sorted(leakage_columns.intersection(features))

if leakage_found:
    raise ValueError(f'데이터 누수 위험 컬럼이 포함되었습니다: {leakage_found}')

print('사용 feature:', features)
print('데이터 누수 컬럼 없음')


사용 feature: ['age', 'item_count', 'total_quantity', 'order_amount', 'order_month', 'order_dayofweek', 'days_since_signup', 'gender', 'city', 'payment_method']
데이터 누수 컬럼 없음


## 6. train, validation, test 분리

모델과 임계값은 validation에서 선택하고, test는 최종 평가에 한 번만 사용합니다.


In [6]:
from src.classification import (
    split_train_validation_test,
    build_split_summary,
)

(
    X_train,
    X_valid,
    X_test,
    y_train,
    y_valid,
    y_test,
    features,
) = split_train_validation_test(
    model_data=model_data,
    numeric_features=numeric_features,
    categorical_features=categorical_features,
    random_state=42,
)

split_summary = build_split_summary(y_train, y_valid, y_test)

print('X_train:', X_train.shape)
print('X_valid:', X_valid.shape)
print('X_test:', X_test.shape)
display(split_summary)


X_train: (148, 10)
X_valid: (50, 10)
X_test: (50, 10)


,split,is_cancelled,class_label,count,ratio
0,train,0,completed,110,0.7432
1,train,1,cancelled,38,0.2568
2,validation,0,completed,37,0.7400
3,validation,1,cancelled,13,0.2600
4,test,0,completed,37,0.7400
5,test,1,cancelled,13,0.2600


## 7. 기준 모델과 학습 모델 비교

가장 많은 클래스만 예측하는 Dummy 모델을 기준으로 Logistic Regression과 Random Forest의 validation 성능을 비교합니다.


In [7]:
from src.classification import train_and_compare_on_validation

(
    models,
    validation_comparison,
    validation_predictions,
    validation_probabilities,
) = train_and_compare_on_validation(
    X_train=X_train,
    X_valid=X_valid,
    y_train=y_train,
    y_valid=y_valid,
    numeric_features=numeric_features,
    categorical_features=categorical_features,
    random_state=42,
)

validation_comparison


,model,evaluation_split,accuracy,precision,recall,f1
0,Logistic Regression,validation,0.66,0.388889,0.538462,0.451613
1,Random Forest,validation,0.76,1.000000,0.076923,0.142857
2,Dummy Most Frequent,validation,0.74,0.000000,0.000000,0.000000


## 8. validation에서 모델과 임계값 선택

Dummy 모델은 기준선이므로 실제 선택 후보에서는 제외합니다. validation의 f1-score를 우선하고 recall과 precision을 함께 확인합니다.


In [8]:
from src.classification import threshold_metrics, choose_threshold

non_dummy_comparison = validation_comparison[
    validation_comparison['model'] != 'Dummy Most Frequent'
]

selected_model_name = str(non_dummy_comparison.iloc[0]['model'])
selected_model = models[selected_model_name]
validation_proba = validation_probabilities[selected_model_name]

threshold_df = threshold_metrics(y_valid, validation_proba)
selected_threshold = choose_threshold(threshold_df)

print('선택 모델:', selected_model_name)
print('선택 임계값:', selected_threshold)
display(threshold_df.sort_values(['f1', 'recall'], ascending=False).head(10))


선택 모델: Logistic Regression
선택 임계값: 0.2


,threshold,accuracy,precision,recall,f1
0,0.20,0.38,0.295455,1.000000,0.456140
9,0.65,0.76,0.555556,0.384615,0.454545
6,0.50,0.66,0.388889,0.538462,0.451613
1,0.25,0.40,0.292683,0.923077,0.444444
4,0.40,0.58,0.333333,0.615385,0.432432
7,0.55,0.68,0.400000,0.461538,0.428571
3,0.35,0.50,0.300000,0.692308,0.418605
2,0.30,0.44,0.285714,0.769231,0.416667
5,0.45,0.60,0.333333,0.538462,0.411765
8,0.60,0.70,0.416667,0.384615,0.400000


## 9. test 데이터 최종 평가

이제 선택이 끝났으므로 test 데이터에서 최종 성능을 한 번 확인합니다.


In [9]:
from src.classification import (
    final_test_evaluation,
    confusion_matrix_dataframe,
    classification_report_dataframe,
)

y_pred_test, y_proba_test, test_metrics = final_test_evaluation(
    selected_model,
    X_test,
    y_test,
    threshold=selected_threshold,
)

confusion_df = confusion_matrix_dataframe(y_test, y_pred_test)
classification_report_df = classification_report_dataframe(y_test, y_pred_test)

display(test_metrics)
display(confusion_df)
display(classification_report_df)


,evaluation_split,threshold,selection_status,accuracy,precision,recall,f1,FP,FN
0,test,0.2,frozen_before_test,0.28,0.244444,0.846154,0.37931,34,2


,pred_completed,pred_cancelled
actual_completed,3,34
actual_cancelled,2,11


,label,precision,recall,f1-score,support
0,completed,0.600000,0.081081,0.142857,37.00
1,cancelled,0.244444,0.846154,0.379310,13.00
2,accuracy,0.280000,0.280000,0.280000,0.28
3,macro avg,0.422222,0.463617,0.261084,50.00
4,weighted avg,0.507556,0.280000,0.204335,50.00


혼동행렬은 다음처럼 읽습니다.

| 구분 | 의미 |
|---|---|
| True Negative | 실제 완료 주문을 완료로 예측 |
| False Positive | 실제 완료 주문을 취소로 잘못 예측 |
| False Negative | 실제 취소 주문을 완료로 잘못 예측 |
| True Positive | 실제 취소 주문을 취소로 예측 |


## 10. 개인정보를 제외한 예측 결과 생성

고객명, 이메일, 주소 같은 불필요한 식별정보는 결과 파일에 저장하지 않습니다.


In [10]:
from src.classification import create_prediction_result

prediction_result = create_prediction_result(
    source_index=X_test.index,
    y_test=y_test,
    y_pred=y_pred_test,
    y_proba=y_proba_test,
    model_name=selected_model_name,
    threshold=selected_threshold,
)

prediction_result.head()


,record_id,source_index,actual_is_cancelled,predicted_is_cancelled,cancel_probability,model,threshold
0,test_0001,28,0,1,0.576120,Logistic Regression,0.2
1,test_0002,146,0,1,0.462559,Logistic Regression,0.2
2,test_0003,211,0,1,0.406685,Logistic Regression,0.2
3,test_0004,125,0,1,0.829364,Logistic Regression,0.2
4,test_0005,77,0,1,0.883050,Logistic Regression,0.2


## 11. LLM 코드 검토 체크리스트


In [11]:
from src.classification import build_classification_checklist

classification_checklist = build_classification_checklist()
classification_checklist


,check_item,status
0,completed와 cancelled만 사용해 이진 타깃을 만들었는가?,□
1,refunded 등 다른 상태를 0 클래스에 섞지 않았는가?,□
2,예측 시점과 feature 가용성 가정을 설명할 수 있는가?,□
3,line_total = quantity × unit_price를 검증했는가?,□
4,병합에 validate를 사용하고 미매칭 0건을 확인했는가?,□
5,"order_status, target, ID, 사후 정보를 feature에서 제외했는가?",□
6,"train, validation, test를 분리했는가?",□
7,모델과 threshold는 validation에서 선택했는가?,□
8,모델과 threshold를 고정한 뒤 test를 한 번만 사용했는가?,□
9,Dummy 기준 모델과 비교했는가?,□


## 12. 전체 파이프라인 실행과 산출물 저장

위 과정을 한 번에 다시 실행해 CSV와 Markdown 보고서를 생성합니다.


In [12]:
from src.classification import run_classification_analysis

classification_result = run_classification_analysis(
    processed_dir=PROCESSED_DIR,
    report_dir=REPORT_DIR,
    random_state=42,
)

print('선택 모델:', classification_result['selected_model_name'])
print('선택 임계값:', classification_result['selected_threshold'])
display(classification_result['test_metrics'])

for name, path in classification_result['output_paths'].items():
    print(name, path, 'OK' if path.exists() else 'MISSING')


선택 모델: Logistic Regression
선택 임계값: 0.2


,model,evaluation_split,threshold,selection_status,accuracy,precision,recall,f1,FP,FN
0,Logistic Regression,test_final,0.2,frozen_before_test,0.28,0.244444,0.846154,0.37931,34,2


model_data_internal C:\Users\ojh23\Documents\projects\llm-data-analysis-course\reports\ch10_classification_model_data_internal.csv OK
target_distribution C:\Users\ojh23\Documents\projects\llm-data-analysis-course\reports\ch10_target_distribution.csv OK
feature_audit C:\Users\ojh23\Documents\projects\llm-data-analysis-course\reports\ch10_feature_audit.csv OK
merge_checks C:\Users\ojh23\Documents\projects\llm-data-analysis-course\reports\ch10_merge_checks.csv OK
data_quality_checks C:\Users\ojh23\Documents\projects\llm-data-analysis-course\reports\ch10_data_quality_checks.csv OK
split_summary C:\Users\ojh23\Documents\projects\llm-data-analysis-course\reports\ch10_split_summary.csv OK
validation_comparison C:\Users\ojh23\Documents\projects\llm-data-analysis-course\reports\ch10_validation_model_comparison.csv OK
threshold_metrics C:\Users\ojh23\Documents\projects\llm-data-analysis-course\reports\ch10_validation_threshold_metrics.csv OK
test_metrics C:\Users\ojh23\Documents\projects\llm-dat

## 13. LLM 검토 프롬프트 예시

```text
온라인 쇼핑몰 주문 취소 분류 코드를 검토해 주세요.

타깃 기준:
- completed=0
- cancelled=1
- refunded와 기타 상태는 제외

검토 항목:
1. order_status와 is_cancelled가 feature에서 제외되었는가?
2. 병합에 validate와 indicator가 사용되었는가?
3. train, validation, test가 분리되었는가?
4. 모델과 임계값은 validation에서 선택했는가?
5. test는 최종 평가에 한 번만 사용했는가?
6. Dummy 기준 모델보다 나은가?
7. accuracy, precision, recall, f1을 함께 확인했는가?
8. 예측 결과에 개인정보가 포함되지 않았는가?
9. 모델 결과를 취소 원인으로 단정하지 않았는가?

필수 수정과 권장 개선을 구분해 설명해 주세요.
```


## 14. 정리

이번 장에서는 완료 주문과 취소 주문을 명확하게 구분하고, 병합 검증, 데이터 누수 방지, train/validation/test 분리, 기준 모델 비교, validation 기반 모델·임계값 선택, 최종 test 평가를 수행했습니다.

다음 장에서는 LLM을 활용해 분석 질문과 프롬프트를 더 구체적으로 만드는 방법으로 이어집니다.


# Chapter 10 답안 양식. 분류 분석으로 주문 취소 여부 예측하기

## 제출 정보

- 이름: 오병희
- GitHub ID: OBHAR
- 작성일: 2026-11-13
- 최종 Notebook URL: https://github.com/OBHAR/llm-data-analysis-study/blob/main/chapter10/chapter10.ipynb

## 1. 분류 문제 정의

- 타깃: `is_cancelled`
- 0의 의미: 완료 주문(`completed`)
- 1의 의미: 취소 주문(`cancelled`)
- 제외한 상태: `refunded` 52건 및 기타 상태
- 예측 단위: 주문 1건
- 예측 시점: 주문 접수 직후, 취소 여부가 확정되기 전으로 가정
- 예측 시점에 사용할 수 있는 정보: 주문 접수 시 확인 가능한 고객 연령·성별·도시, 결제 방식, 주문일의 월·요일, 주문 상품 수·수량·금액 및 가입 후 경과일
- 예측 이후에만 알 수 있어 제외한 정보: `order_status`, `is_cancelled`, 취소 사유와 취소 시각 등 결과가 확정된 뒤 생성되는 정보. 주문·고객·상품 식별자도 Feature에서 제외

### 나의 해석과 판단

주문 1건의 취소 여부를 예측하도록 정의하고, 완료와 취소만 이진 타깃으로 사용했다. `refunded`는 완료나 취소 어느 쪽에도 임의로 넣지 않았다. 예측 시점은 주문 접수 직후로 가정했으며, 장바구니 특징이 그 시점에 이미 확정되어 있다는 전제가 필요하다. `order_status`처럼 결과를 직접 나타내는 값이나 취소 후 생성되는 정보는 예측에 사용할 수 없다.

---

## 2. Feature Contract와 Leakage Audit

### 사용한 숫자형 Feature

`age`, `item_count`, `total_quantity`, `order_amount`, `order_month`, `order_dayofweek`, `days_since_signup`

### 사용한 범주형 Feature

`gender`, `city`, `payment_method`

### 금지 Feature

```text
order_status
is_cancelled
order_id
customer_id
product_id
취소 이후 정보
```

### 나의 해석과 판단

원본 파일에 컬럼이 있다는 이유만으로 예측 시점에 사용할 수 있는 것은 아니다. 해당 정보가 주문 접수 때 실제로 확보되는지 따져야 한다. `signup_date`는 `days_since_signup`을 계산하는 데만 사용하고 모델 입력에서는 제외했다. 취소 여부와 연결되는 상태값, 식별자, 사후 정보는 Feature Audit에서 제외된 것을 확인했다.

---

## 3. 주문 단위 특징과 병합 검증

### 계산 관계 확인

```text
line_total = quantity × unit_price
```

- 계산 불일치 건수: 0건 (주문 상세 764행 확인)
- 주문 특징 `order_id` 중복 건수: 0건
- 가입일이 주문일보다 늦은 분류 대상 주문: 45건
- 날짜 역전 처리: 해당 행을 삭제하지 않고 `days_since_signup`만 결측 처리
- 처리 전후 분류 대상 행 수: 248건 → 248건

### 병합 검증

| 병합         | 관계          | 병합 전 행 수 | 병합 후 행 수 | 미매칭 | 판정   |
| ---------- | ----------- | -------: | -------: | --: | ---- |
| 주문 ↔ 주문 특징 | one_to_one  |      248 |      248 |   0 | PASS |
| 주문 ↔ 고객    | many_to_one |      248 |      248 |   0 | PASS |

### 결과 관찰

주문 상세를 주문 단위로 집계해 `item_count`, `total_quantity`, `order_amount`를 만들었다. 두 병합 모두 행 수가 유지됐고, 미매칭은 없었다. 합성 데이터에서 발생한 날짜 역전은 전체 주문에서는 56건, 이진 분류 대상에서는 45건이었다. 원본 CSV는 수정하지 않았다.

### 나의 해석과 판단

미매칭 값을 원인 확인 없이 0으로 채우면 실제로 특징이 0인 주문과 연결 오류가 난 주문을 구분할 수 없다. 이번에는 두 관계 검증에서 미매칭이 0건이라 추가 대체 처리는 필요하지 않았다. 행 삭제나 원본 CSV 수정은 하지 않았다. 실행 결과에서 대상 248건 유지와 `days_since_signup` 결측 45건을 확인했으며, 숫자형 Pipeline의 중앙값 대체는 Train 데이터에서 학습된다. 모델과 Threshold도 LLM 답변만으로 정하지 않고 Validation 출력으로 확인했다.

---

## 4. 클래스 비율

| 클래스 | 의미        |  개수 |     비율 |
| --: | --------- | --: | -----: |
|   0 | completed | 184 | 74.19% |
|   1 | cancelled |  64 | 25.81% |

![노트북 실행 결과: 데이터 품질·병합 검증과 클래스 비율](images/evidence_01_data_quality_target.png)

### 결과 관찰

분류 대상 248건 중 완료 주문이 약 74.19%, 취소 주문이 약 25.81%였다. `refunded` 52건은 분류 대상에서 제외했다.

### 나의 해석과 판단

다수 클래스인 완료만 예측해도 Accuracy가 약 0.74가 될 수 있으므로 Accuracy만으로 취소 예측 성능을 판단하기 어렵다. 취소 주문을 실제로 찾았는지 Precision, Recall, F1과 함께 확인해야 한다.

---

## 5. Dummy Baseline과 Validation 모델 비교

| 모델                  | Accuracy | Precision | Recall |    F1 |
| ------------------- | -------: | --------: | -----: | ----: |
| Dummy Most Frequent |    0.740 |     0.000 |  0.000 | 0.000 |
| Logistic Regression |    0.660 |     0.389 |  0.538 | 0.452 |
| Random Forest       |    0.760 |     1.000 |  0.077 | 0.143 |

![노트북 실행 결과: Validation 모델 비교](images/evidence_02_model_comparison.png)

- Validation에서 선택한 모델: **Logistic Regression**

### 결과 관찰

모델 비교 표의 분류 결과는 기본 Threshold 기준이다. Random Forest는 Accuracy 0.76으로 가장 높았지만 취소 Recall은 0.077이었다. Dummy는 완료만 예측하여 Accuracy 0.74를 얻었지만 취소를 전혀 찾지 못했다.

### 나의 해석과 판단

Logistic Regression은 Dummy보다 Accuracy는 낮지만, 취소를 찾아 Recall 0.538과 F1 0.452를 얻었다. 취소 탐지 성능을 함께 고려해 Validation에서 Logistic Regression을 선택했다. Accuracy가 가장 높은 모델을 자동으로 최종 선택하지 않았다.

### 한계와 추가 확인 사항

Validation 취소 표본은 13건이라 지표가 몇 건의 예측에 크게 영향을 받을 수 있다. 작은 Validation 표본에서 얻은 차이를 안정적인 일반화 성능으로 단정할 수 없다.

---

## 6. Train / Validation / Test 역할

| Split      | 역할              | 클래스 0 건수 | 클래스 1 건수 |
| ---------- | --------------- | -------: | -------: |
| Train      | 모델 학습           |      110 |       38 |
| Validation | 모델·Threshold 선택 |       37 |       13 |
| Test       | 선택 이후 최종 평가     |       37 |       13 |

### 나의 해석과 판단

Test를 모델이나 Threshold 선택에 사용하면 Test 성능에 맞춰 선택이 이루어져 최종 평가가 낙관적으로 치우칠 수 있다. 따라서 모델과 Threshold를 Validation에서 고정한 뒤 Test는 최종 평가에만 사용했다.

### 교육용 random split의 한계

무작위 분할은 과거·미래 순서를 반영하지 않는다. 실제 미래 주문에 적용하기 전에는 과거 주문으로 학습하고 이후 기간으로 평가하는 시간순 분할과 별도의 out-of-time 검증이 필요하다.

---

## 7. Validation 성능과 우선 지표

선택된 Logistic Regression과 Threshold 0.20의 Validation 성능입니다.

- Accuracy: 0.380
- Precision: 0.295
- Recall: 1.000
- F1: 0.456

### 내가 가장 중요하게 본 지표

Recall과 F1을 우선해 살펴봤다.

### 그 이유

이번 분석의 목적은 취소 위험 주문을 찾아내는 것이므로 실제 취소 주문을 놓치는 비율을 확인할 필요가 있다. 다만 Threshold 0.20에서는 Recall을 높이는 대신 오탐이 늘어 Precision이 낮아졌다. 실제 FP/FN 업무 비용을 알 수 없어 이 우선순위는 과제 목적에 따른 잠정 판단이다.

> 실제 FP/FN 비용 정보가 없어 취소 위험을 놓치는 FN을 더 부담스럽게 보는 가정을 두었지만, 운영 비용으로 확인된 사실은 아니다.

---

## 8. Threshold 비교

| Threshold | Precision | Recall |    F1 | 예상 특징                     |
| --------: | --------: | -----: | ----: | ------------------------- |
|      0.20 |     0.295 |  1.000 | 0.456 | 취소를 모두 찾지만 완료 주문 오탐이 많음   |
|      0.50 |     0.389 |  0.538 | 0.452 | Precision·Recall이 중간 수준   |
|      0.65 |     0.556 |  0.385 | 0.455 | 오탐은 줄 수 있지만 취소를 더 놓칠 수 있음 |

![노트북 실행 결과: Threshold별 Validation 지표](images/evidence_03_threshold.png)

### 내가 선택한 Threshold

**0.20**

### 선택 이유

Validation에서 가장 높은 F1(0.456)을 보였다. 0.65의 F1(0.455)과 차이는 매우 작지만, 0.20은 Recall 1.000으로 Validation의 취소 주문 13건을 모두 찾아냈다. 이 선택은 FN을 줄이는 쪽에 무게를 둔 것이다.

### 나의 해석과 판단

Threshold를 Test 결과로 고르면 Test가 모델 선택에 사용되어 독립적인 최종 평가가 아니게 된다. 따라서 모델과 Threshold를 Validation에서 결정하고, 고정한 선택을 Test에 적용했다.

---

## 9. FP/FN 해석

- FP 의미: 실제 완료 주문을 취소 위험으로 잘못 예측한 경우
- FN 의미: 실제 취소 주문을 완료로 잘못 예측한 경우
- 현재 목적에서 더 부담스러울 수 있는 오류: 취소 위험 조기 탐지가 목적이라는 가정에서는 FN을 더 부담스럽게 볼 수 있다. 다만 실제 비용 자료가 없어 잠정 판단이다.
- 그 판단에 추가로 필요한 업무 정보: 잘못된 위험 표시로 인한 확인·보류 비용, 취소로 인한 손실, 사전 대응의 성공률, 고객 불편 및 이탈 비용

![노트북 실행 결과: Final Test 지표와 Confusion Matrix](images/evidence_04_final_test_confusion.png)

### 업무·분석적 의미

Final Test에서 FP는 34건, FN은 2건이었다. FP가 발생하면 완료 주문에 불필요한 확인 연락이나 검토·보류가 발생할 수 있다. FN은 실제 취소 주문을 사전에 찾지 못해 대응 기회를 놓칠 수 있다. 이번 결과에서는 오탐 수가 특히 많아, FN을 줄이는 이점만으로 Threshold 0.20을 운영에 바로 적용하기 어렵다.

### 한계와 추가 확인 사항

수업 데이터에는 실제 업무 비용, 취소 사유, 대응 조치 결과가 없다. 따라서 어느 오류가 금액상 더 큰 손실인지 확정할 수 없다. 비용 정보를 확보한 뒤 Threshold를 다시 설계해야 한다.

---

## 10. Final Test

- 고정한 모델: Logistic Regression
- 고정한 Threshold: 0.20
- Accuracy: 0.280
- Precision: 0.244
- Recall: 0.846
- F1: 0.379
- Validation 대비 변화: Accuracy -0.100, Precision -0.051, Recall -0.154, F1 -0.077

### 결과 관찰

Final Test 혼동행렬에서 실제 완료 37건 중 3건은 완료로, 34건은 취소 위험으로 예측했다. 실제 취소 13건 중 11건은 찾아냈고 2건은 놓쳤다. Validation보다 모든 지표가 낮아졌다.

### 나의 해석과 판단

취소 Recall은 0.846으로 비교적 높았지만 Precision 0.244, Accuracy 0.280, F1 0.379이며 FP가 34건이었다. Final Test 표본이 50건으로 작아 성능 추정의 불확실성도 크다. 현재 결과만으로 미래 주문에 대한 일반화나 운영 적합성을 판단하기 어렵다. Test 결과를 확인한 뒤 모델과 Threshold는 변경하지 않았다.

### 선택 보호 확인

- [x] Test 결과를 보기 전에 모델을 고정했습니다.
- [x] Test 결과를 보기 전에 Threshold를 고정했습니다.
- [x] Test 결과를 본 뒤 같은 Test를 이용해 선택을 바꾸지 않았습니다.

---

## 11. Internal / Public 결과 구분

### 공개 Prediction 컬럼

```text
record_id
actual_is_cancelled
predicted_is_cancelled
cancel_probability
model
threshold
```

### 공개 결과에 없어야 할 컬럼

```text
source_index
order_id
customer_id
product_id
```

### 결과 관찰

Internal 결과에는 오류를 원본 행과 대조할 수 있도록 `source_index`가 포함될 수 있지만, 공개 Prediction에는 포함하지 않았다. 자동 생성한 공개 결과는 위 여섯 컬럼으로 제한됐다.

### 나의 해석과 판단

내부 오류 분석은 원본 데이터 추적이 필요할 수 있지만, 공개 파일에는 주문·고객·상품 식별자를 남길 필요가 없다. 분석 추적에 필요한 정보와 외부에 공개할 결과를 분리해 불필요한 식별 정보 노출을 줄였다.

---

## 12. Validation Evidence

다음 Evidence 중 확인한 항목에 체크합니다.

- [x] `ch10_target_distribution.csv`
- [x] `ch10_feature_audit.csv`
- [x] `ch10_merge_checks.csv`
- [x] `ch10_data_quality_checks.csv`
- [x] `ch10_split_summary.csv`
- [x] `ch10_validation_model_comparison.csv`
- [x] `ch10_validation_threshold_metrics.csv`
- [x] `ch10_test_metrics.csv`
- [x] `ch10_confusion_matrix.csv`
- [x] `ch10_classification_validation.csv`

### 자동 Validation 결과

![노트북 실행 결과: 자동 Validation 8개 항목 PASS](images/evidence_05_validation_pass.png)

`ch10_classification_validation.csv`의 8개 검증 항목이 모두 PASS였다. 타깃 정의, 금지 Feature 제외, 병합 행 수·미매칭, 모든 Split의 두 클래스 존재, Validation 기반 모델·Threshold 선택, 공개 Prediction 식별자 제거, 고정된 모델·Threshold로 Test를 최종 평가했는지 확인했다.

### 자동 PASS가 의미하지 않는 것

자동 PASS는 코드의 분석 절차와 데이터 계약이 지정된 조건을 통과했다는 뜻이다. 운영 적합성, 공정성, 미래 기간의 성능 유지, 실제 오류 비용까지 검증한 것은 아니다. 이번 Test 표본이 작고 성능도 낮아 별도 검증이 필요하다.

---

## 13. 최종 사용 판단

- [ ] 참고용 사용 가능
- [ ] 추가 검증 후 사용 가능
- [x] 현재 사용 보류

### 판단 근거

1. 취소 주문은 전체 분류 대상의 25.81%다. Dummy는 Accuracy 0.740이지만 취소를 전혀 찾지 못해 Accuracy만으로 모델을 판단할 수 없다.
2. Validation에서 고른 Logistic Regression·Threshold 0.20 조합은 Test에서 Precision 0.244, Recall 0.846, F1 0.379를 보였고 FP 34건이 발생했다.
3. 실제 FP/FN 비용이 없고 Test 표본도 50건뿐이며, 무작위 분할만으로 미래 주문에서의 성능을 보장할 수 없다.

### 현재 모델의 가장 큰 위험

취소를 놓치지 않도록 낮춘 Threshold에서 완료 주문 37건 중 34건을 취소 위험으로 잘못 표시했다. 이 수준의 오탐은 불필요한 검토를 크게 늘릴 수 있다.

### 다음 개선 우선순위

1. FP와 FN의 실제 비용 및 대응 조치 성공률을 수집해 업무 기준으로 Threshold를 정한다.
2. 시간순 분할과 이후 기간 데이터의 out-of-time 검증으로 미래 주문 성능을 확인한다.
3. 더 많은 실제 주문으로 재평가하고, 예측 시점에 Feature가 실제 사용 가능한지 업무 흐름과 대조한다.

### 예측과 인과 구분

현재 확인한 것은 입력 Feature와 취소 여부 사이에서 나타난 예측 패턴이다. 이 결과만으로 특정 Feature가 주문 취소를 일으켰다고 주장하거나 취소 원인을 설명할 수는 없다.

---

## 최종 체크

- [x] `completed=0`, `cancelled=1` 타깃 범위를 정의했습니다.
- [x] `refunded`와 기타 상태를 제외했습니다.
- [x] 예측 시점과 Feature Contract를 정의했습니다.
- [x] target, ID, 사후 정보를 feature에서 제외했습니다.
- [x] `line_total = quantity × unit_price`를 검증했습니다.
- [x] 주문 단위 특징과 병합 관계를 검증했습니다.
- [x] 클래스 비율을 확인했습니다.
- [x] Dummy baseline과 비교했습니다.
- [x] Train / Validation / Test 역할을 구분했습니다.
- [x] 모델을 Validation에서 선택했습니다.
- [x] Threshold를 Validation에서 선택했습니다.
- [x] Final Test 전에 선택을 고정했습니다.
- [x] Precision / Recall / F1을 해석했습니다.
- [x] FP/FN의 의미를 설명했습니다.
- [x] Public prediction에서 식별자를 제거했습니다.
- [x] Validation Evidence를 확인했습니다.
- [x] random split의 한계를 기록했습니다.
- [x] 예측 패턴을 원인처럼 단정하지 않았습니다.
- [x] 최종 Notebook URL을 GitHub 탑재 후 입력합니다.


## 14. 4번 실행 오류 확인과 날짜 역전 처리

### 오류가 발생한 상황

4번 셀에서 `build_classification_dataset()`을 실행할 때 `signup_date`가 `order_date`보다 늦은 주문이 45건 있다는 `ValueError`가 발생했다. 원본 주문일과 가입일을 확인하였고, 수업용 합성 데이터를 만들 때 두 날짜를 독립적으로 생성하면서 생긴 데이터 문제임을 확인했다. 이는 분석 코드가 잘못된 주문을 임의로 만든 것이 아니라, 제공 데이터의 날짜 정합성 문제다.

### 선택한 처리 방법

날짜 역전 주문을 삭제하지 않고 해당 주문의 파생 변수 `days_since_signup`만 `NaN`으로 처리했다. 전체 주문의 날짜 역전은 56건이고, 이진 분류 대상(`completed`, `cancelled`)은 45건이다. 나머지 11건은 `refunded`로 분류 분석에서 제외했다.

### 코드 변경과 재실행 확인

`src/classification.py`에서 날짜 역전 시 오류를 발생시키던 검증을 처리 정책에 맞게 변경해, 음수 경과일만 결측으로 처리하고 품질 점검표에 건수를 기록하도록 했다. 원본 CSV는 수정하지 않았다. 처리 전후 분류 데이터는 모두 248건이며, `days_since_signup` 결측 45건은 숫자형 Pipeline의 `SimpleImputer(strategy="median")`가 학습 데이터에서 계산한 중앙값으로 대체한다.

수정 후 Notebook을 다시 실행해 4번 셀을 포함한 분석 흐름이 완료되는 것을 확인했다. 병합은 각각 248행을 유지했고 미매칭은 0건이었다. 자동 Validation의 8개 항목도 모두 PASS였다. 이후 모델과 Threshold는 Validation에서 선택하고, 고정한 설정으로 Test를 최종 평가했다.

### 처리의 한계

결측 대체는 실제 가입 경과일을 복원하지 않는다. 날짜 역전 45건에서는 이 Feature 정보를 활용하지 못하며, 대체된 중앙값을 실제 경과일이라고 해석할 수 없다. 따라서 결과는 제공된 합성 데이터와 현재 처리 정책에 한정된다. 실제 운영 전에는 날짜 생성·수집 규칙과 예측 시점의 Feature 가용성을 확인해야 한다.